# Telemetry-source valuation and leakage-audited explainable alerts on RT-IoT2022

Runs every experiment of the article end to end: leakage audit, two-pass detector, Shapley attribution over telemetry sources, degraded telemetry with fallback models, review routing, and figures.

* Dataset: read from `MyDrive/Datasets/RT-IoT2022/rt-iot2022.csv` if present (read only); otherwise downloaded from a public mirror to `/content` and checked against its SHA-256.
* Heavy intermediate files stay on `/content`; result tables and figures are copied to `MyDrive/Outputs/RTIoT_Telemetry/`.
* Every long step prints progress.


In [ ]:
import os, sys, hashlib, shutil, subprocess, urllib.request
REPO = 'https://raw.githubusercontent.com/hamamh66/IoT-Telemetry-Source-Valuation/main/'
WORK = '/content/rtiot' if os.path.isdir('/content') else os.path.abspath('rtiot_work')
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
except Exception as e:
    print('Google Drive not available:', e); DRIVE = None
SHA = '956956c09c1764584fa08acd0f6876475626bcedcd6a6b1f8c492c2e9a2089ea'
src = os.path.join(DRIVE, 'Datasets/RT-IoT2022/rt-iot2022.csv') if DRIVE else None
DATA = os.path.join(WORK, 'rt-iot2022.csv')
if src and os.path.exists(src):
    print('using dataset from Drive:', src); DATA = src
elif not os.path.exists(DATA):
    url = 'https://huggingface.co/datasets/michaelmallari/rt-iot2022/resolve/main/rt-iot2022.csv'
    print('downloading RT-IoT2022 from', url); urllib.request.urlretrieve(url, DATA)
h = hashlib.sha256(open(DATA, 'rb').read()).hexdigest()
print('SHA-256', h); assert h == SHA, 'dataset file differs from the one used in the article'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True)
for f in ('rtiot_xai.py', 'run_rtiot.py', 'review_routing.py', 'make_figures.py'):
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f); print('fetched', f)


## 1. All experiments over five stratified splits (about 15 min on a Colab CPU)


In [ ]:
def run(*args):
    p = subprocess.Popen([sys.executable, '-u', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    assert p.wait() == 0, f'{args[0]} failed'
OUT = os.path.join(WORK, 'Outputs')
run('run_rtiot.py', DATA, OUT)


## 2. Review routing and figures


In [ ]:
run('review_routing.py', DATA, OUT)
run('make_figures.py', OUT)


## 3. Results and backup of the small result files to Drive


In [ ]:
import pandas as pd
from IPython.display import display, Image
T = os.path.join(OUT, 'Tables')
for f in ('audit_summary.csv', 'compact_summary.csv', 'stream_shapley.csv', 'degraded_summary.csv', 'review_routing_summary.csv'):
    print(f); display(pd.read_csv(os.path.join(T, f)).round(4))
for f in sorted(os.listdir(os.path.join(OUT, 'Figures'))):
    display(Image(os.path.join(OUT, 'Figures', f), width=720))
if DRIVE:
    dst = os.path.join(DRIVE, 'Outputs/RTIoT_Telemetry')
    for sub in ('Tables', 'Figures', 'Others'):
        os.makedirs(os.path.join(dst, sub), exist_ok=True)
        for f in os.listdir(os.path.join(OUT, sub)):
            if not f.endswith('.pkl'):
                shutil.copy(os.path.join(OUT, sub, f), os.path.join(dst, sub, f))
    print('small result files copied to', dst)
